In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
%load_ext autoreload
%autoreload 2

import sys
import os

chemin_racine = os.path.abspath('..')

if chemin_racine not in sys.path:
    sys.path.append(chemin_racine)

# 5. Mise en place du modèle XGBoost

Ici, nous avons pour but principal d'essayer de constituer un modèle XGBoost qui saura justifier de meilleures performances que la régression logistique qui nous sert de baseline.

## a. Import initial et setup de la cross-validation du modèle

Les modèles XGBoost étant des modèles boîte noire plus complexes qu'un modèle transparent comme une régression logistique, il est plus difficile de juger d'un surapprentissage ou d'un coup de chance. Il convient donc que l'on mette en place une validation croisée pour minimiser les chances de bruit causé par l'aléatoire et le surapprentissage. Nous choisissons d'utiliser une `StratifiedKFold` pour pouvoir garder le ratio bons payeurs / mauvais payeurs dans les splits. Puisque l'on avait choisi de prendre 20% du dataset en validation set, on va choisir de prendre 5 splits pour la cross-validation (on aura à chaque fois 4 splits pour train et le dernier pour test)

In [5]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

raw_data_dir = "../data/raw/"
raw_train_val_file = 'cs-training.csv'
raw_test_file = 'cs-test.csv'

credit_data = pd.read_csv(raw_data_dir + raw_train_val_file, index_col=0, header=0)
X = credit_data.drop("SeriousDlqin2yrs", axis=1)
y = credit_data["SeriousDlqin2yrs"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=67)

## b. Benchmark de la baseline avec cross-validation :

Pour avoir une mesure totalement équitable, on fait le choix de remesurer les performances de la Régression Logistique qui nous sert de baseline en utilisant la même validation croisée que l'on utilisera pour les modèles XGBoost. Nous allons alors comparer les scores OOF de chacun pour avoir nos résultats. On rappelle que le score OOF consiste à grouper les évaluations faites sur les splits du dataset par les modèles en ayant uniquement connaissance des 4 autres splits.

On décide de faire une boucle manuelle pour les scores, établie dans une fonction dans `utils.py`.

Puisqu'on décide d'utiliser la baseline précédente ici, on en profite pour implémenter des fonctions qui font de l'import et de l'export de modèles en joblib.

In [8]:
from src.utils import evaluate_model_oof, display_model_accuracy, load_pipeline

baseline_pipeline = load_pipeline("logreg_baseline_v2.joblib")

auc_roc_baseline, gini_baseline, oof_baseline = evaluate_model_oof(baseline_pipeline, X, y,skf)

display_model_accuracy(auc_roc_baseline,gini_baseline)


Performances du modèle : AUC-ROC : 0.856 
Gini : 0.713 (Soit 71.29%)


La baseline établie au notebook 4 avec notre approche OOP donne un **AUC-ROC à 0.856** et un **coefficient de Gini à 0.713** sur **l'intégralité du dataset** en OOF. Pour comparer, sur **20% du dataset** précédemment, on avait un **AUC-ROC à 0.858** et un **coefficient de Gini à 0.716**. On a donc des performances très similaires en score pur sur un nombre de clients beaucoup plus conséquent. Nous prendrons donc ces nouveaux chiffres comme référence pour la suite.

## c. Simple XGBoost

Dans cette section, nous allons commencer par entraîner et mesurer l'efficacité d'un modèle XGBoost sans fine tuner les paramètres pour se faire une idée de la performance de ce genre de modèles comparée à celle de notre baseline out of the box.



In [9]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from src.pipelines import get_cleaner_pipeline

preprocessor = get_cleaner_pipeline()
simple_xgb = XGBClassifier(scale_pos_weight=(y == 0).sum() / (y == 1).sum(), eval_metric='auc', random_state=67)

simple_xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('simple_xgb', simple_xgb)
])

simple_xgb_auc_roc, simple_xgb_gini, simple_xgb_oof = evaluate_model_oof(simple_xgb_pipeline, X, y, skf)
display_model_accuracy(simple_xgb_auc_roc,simple_xgb_gini)

Performances du modèle : AUC-ROC : 0.848 
Gini : 0.695 (Soit 69.54%)


Notre modèle XGBoost out of the box affiche un **AUC-ROC à 0.848** et un **coefficient de Gini à 0.695 (soit 69.54%)**, ce qui constitue un recul de performance par rapport aux **0.856 d'AUC-ROC** et **71.29% de Gini** obtenus par notre baseline logistique en OOF. En effet, s'agissant d'un modèle boîte noire plus complexe qu'une régression linéaire transparente, l'algorithme est naturellement plus exposé au risque de surapprentissage lorsqu'il est appliqué sans fine tuner ses paramètres. Ce premier constat démontre qu'une architecture complexe brute ne surpasse pas automatiquement un modèle simple bien pré-traité et justifie pleinement le recours à notre cross-validation pour mener à bien l'optimisation des hyperparamètres nécessaire afin d'espérer battre notre socle de référence.